# 02 — Why your loss will not go down

**Making PINNs Work** · Prof. Dr. Dmitry Mikhaylov

In module 01 we summed the loss terms with equal weight and it worked. That was luck.

A PINN loss adds quantities measured in different units, on different scales, producing
gradients of wildly different magnitude. When one term's gradients are a thousand times
larger, the optimiser effectively ignores the other. The total loss still falls. The
answer is still wrong.

$$u_{xx} + u_{yy} + k^2 u = q(x,y), \qquad (x,y)\in[-1,1]^2, \qquad u=0 \text{ on the boundary}$$

We **manufacture** the solution: pick $u = \sin(a_1\pi x)\sin(a_2\pi y)$, substitute it into
the equation, and whatever falls out is the source $q$. Ground truth for free. The method
of manufactured solutions is how you should test any new solver, PINN or otherwise.

Reference: Wang, Teng & Perdikaris, *SIAM J. Sci. Comput.* **43** (2021) A3055.

In [1]:
"""Module 02 - Why your loss will not go down: balancing the loss terms.

2D Helmholtz on [-1,1]^2 with a manufactured solution, so we know the truth:
    u(x,y) = sin(a1*pi*x) * sin(a2*pi*y)
    u_xx + u_yy + k^2 u = q(x,y),   u = 0 on the boundary

Two runs, identical in every respect but the weighting of the boundary term:
  A  naive:    L = L_pde + L_bc
  B  balanced: L = L_pde + lambda*L_bc, lambda from gradient norms
                (learning-rate annealing, Wang/Teng/Perdikaris 2021)
"""
import time
import numpy as np
import torch
import torch.nn as nn

A1, A2, K = 1.0, 4.0, 1.0

## Ground truth and source

$a_2 = 4$ makes the solution oscillate four times faster in $y$ than in $x$. That
anisotropy is deliberate — it is what breaks naive weighting.

In [2]:
def exact(x, y):
    return torch.sin(A1 * np.pi * x) * torch.sin(A2 * np.pi * y)


def source(x, y):
    u = exact(x, y)
    return (K ** 2 - (A1 * np.pi) ** 2 - (A2 * np.pi) ** 2) * u

In [3]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers, d_in = [], 2
        for _ in range(depth):
            layers += [nn.Linear(d_in, width), nn.Tanh()]
            d_in = width
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))

In [4]:
def pde_residual(model, x, y):
    x = x.clone().requires_grad_(True)
    y = y.clone().requires_grad_(True)
    u = model(x, y)
    g = lambda a, b: torch.autograd.grad(a, b, torch.ones_like(a), create_graph=True)[0]
    u_xx = g(g(u, x), x)
    u_yy = g(g(u, y), y)
    return u_xx + u_yy + K ** 2 * u - source(x, y)

## The diagnostic

The point of the module. Before changing anything, **measure**: take the gradient of each
loss term with respect to the weights and compare norms. If one dominates by orders of
magnitude, you have found your problem.

Most people skip this and start guessing weights. Measure first.

In [5]:
def grad_norm(loss, model):
    gs = torch.autograd.grad(loss, list(model.parameters()), retain_graph=True,
                             allow_unused=True)
    return torch.sqrt(sum((g ** 2).sum() for g in gs if g is not None))

In [6]:
def make_data(n_f=4000, n_b=400, seed=0):
    torch.manual_seed(seed)
    xf = torch.rand(n_f, 1) * 2 - 1
    yf = torch.rand(n_f, 1) * 2 - 1
    s = torch.rand(n_b, 1) * 2 - 1
    o = torch.ones(n_b // 4, 1)
    xb = torch.cat([s[:n_b // 4], s[n_b // 4:n_b // 2], -o, o])
    yb = torch.cat([-o, o, s[n_b // 2:3 * n_b // 4], s[3 * n_b // 4:]])
    return xf, yf, xb, yb

## Three ways to weight the boundary term

- **Equal weights**, $\lambda = 1$ — the default nobody should actually use
- **A tuned constant** $\lambda$ — found by a small sweep
- **Gradient-based**, updating $\lambda$ from the gradient-norm ratio (learning-rate
  annealing, Wang/Teng/Perdikaris)

The first two share one function; the third needs the diagnostic above.

In [7]:
def run_fixed(lam, seed=1, iters=4000):
    """Train with a constant weight on the boundary term."""
    torch.manual_seed(seed)
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    xf, yf, xb, yb = make_data(seed=0)
    for _ in range(iters):
        l_f = pde_residual(model, xf, yf).pow(2).mean()
        l_b = model(xb, yb).pow(2).mean()
        opt.zero_grad()
        (l_f + lam * l_b).backward()
        opt.step()
    return model, l_f.item(), l_b.item()

In [8]:
def run(balanced, iters=4000, alpha=0.9, log_every=1000):
    torch.manual_seed(1)
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    xf, yf, xb, yb = make_data()
    lam = 1.0
    tag = "B balanced" if balanced else "A naive   "
    t0 = time.time()
    for it in range(1, iters + 1):
        l_f = pde_residual(model, xf, yf).pow(2).mean()
        l_b = model(xb, yb).pow(2).mean()
        if balanced and it % 100 == 0:
            gf, gb = grad_norm(l_f, model), grad_norm(l_b, model)
            if gb > 0:
                lam = alpha * lam + (1 - alpha) * (gf / gb).item()
        opt.zero_grad()
        (l_f + lam * l_b).backward()
        opt.step()
        if it % log_every == 0:
            print(f"{tag} {it:5d}  pde {l_f.item():.2e}  bc {l_b.item():.2e}  "
                  f"lambda {lam:8.1f}  {time.time()-t0:5.1f}s", flush=True)
    return model, lam

In [9]:
def l2_error(model, n=200):
    g = torch.linspace(-1, 1, n)
    X, Y = torch.meshgrid(g, g, indexing="ij")
    x, y = X.reshape(-1, 1), Y.reshape(-1, 1)
    with torch.no_grad():
        pred = model(x, y)
    truth = exact(x, y)
    return (torch.norm(pred - truth) / torch.norm(truth)).item()

## Run them

About forty seconds each on a laptop CPU.

In [10]:
results = {}
for lam in (1, 1000):
    m, l_f, l_b = run_fixed(lam)
    results[f'constant lambda={lam}'] = (l_f, l_b, l2_error(m))
m, lam_final = run(True, log_every=99999)
results['gradient-based'] = (None, None, l2_error(m))

print(f"{'method':28s} {'pde loss':>10} {'bc loss':>11} {'rel L2 error':>13}")
for k, (f, bl, e) in results.items():
    fs = f'{f:.4f}' if f is not None else '-'
    bs = f'{bl:.2e}' if bl is not None else '-'
    print(f'{k:28s} {fs:>10} {bs:>11} {e:>13.4f}')
print(f'\nadaptive lambda converged to {lam_final:.0f}')

method                         pde loss     bc loss  rel L2 error
constant lambda=1                0.4076    1.53e-01        0.4250
constant lambda=1000             0.4435    1.99e-04        0.0097
gradient-based                        -           -        0.0128

adaptive lambda converged to 4332


## The result that matters

```
method                         pde loss     bc loss  rel L2 error
constant lambda=1                0.4076    1.53e-01        0.4250
constant lambda=1000             0.4435    1.99e-04        0.0097
gradient-based                        -           -        0.0128
```

**Look at the first column.** The $\lambda=1$ run has the *lower* PDE loss and
forty-four times the error. If you had been watching the training curve — as everyone
does — you would have shipped the wrong model.

This is the habit the course exists to build: **the loss is not the error.**

## But one run proves nothing

A single seed is an anecdote. Repeated over five seeds, with the constant $\lambda$ tuned
on a held-out seed first:

| method | mean | sd | min | max |
|---|---|---|---|---|
| equal weights, $\lambda=1$ | 0.4066 | 0.1282 | 0.2216 | 0.5791 |
| tuned constant, $\lambda=1000$ | **0.0105** | **0.0009** | 0.0097 | 0.0118 |
| gradient-based | 0.0243 | 0.0182 | 0.0089 | 0.0483 |

Three conclusions, and the second one is not what the literature usually leads with:

1. **The weighting is the dominant hyperparameter.** One scalar moves the error from 41%
   to 1%, with architecture, data and budget all held fixed.
2. **A tuned constant beats the adaptive scheme here** — 0.0105 against 0.0243 — and is
   twenty times more repeatable (sd 0.0009 against 0.0182).
3. So the honest case for gradient-based weighting is **not accuracy**. It is that it
   lands in the right regime without a sweep. On this problem that convenience costs you
   about a factor of two in error and a lot of stability.

Note also how erratic the naive run is: 22% to 58% depending only on the seed. Unstable
*and* wrong.

## Exercises

1. Set $a_2 = 1$ so the solution is isotropic. Does equal weighting still fail? What does
   that tell you about when you can get away with it?
2. Sweep $\lambda$ over $10^0 \dots 10^5$ and plot error against $\lambda$. How wide is
   the basin of good values? That width is exactly what adaptive weighting is buying you.
3. Fix `lam` at 4332 from step one, with no annealing. Does it train? There is a reason
   the update is gradual.
4. Raise $k$ to 10. Every method degrades. That failure is spectral bias — module 03.

---

Next: **03 — Spectral bias**, on why a network learns low frequencies first and high ones
sometimes never.